In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)

Project Root: /Users/dhairyas87/Documents/Projects/explainable-credit-risk-management/xai-credit-risk-freddie-mac


# Model Training

## Objective

This notebook develops predictive models for borrower financial stress.

The objectives are:

- Build a baseline model using traditional mortgage origination features
- Build a BSS-enhanced model using the proposed Borrower Serviceability Score framework
- Compare predictive performance
- Evaluate whether BSS provides incremental predictive value

This notebook begins with a Logistic Regression benchmark model before moving to more advanced machine learning techniques.

In [2]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from src.preprocessing import (
    prepare_xy,
    build_preprocessor
)

# Load Modeling Datasets

The baseline feature store contains traditional mortgage origination characteristics and excludes the proposed BSS framework.

Temporal split:

- Train = 2018Q1 + 2018Q2
- Validation = 2018Q3
- Test = 2018Q4

In [3]:
train_df = pd.read_parquet(
    "../data/modeling/train_baseline.parquet"
)

valid_df = pd.read_parquet(
    "../data/modeling/valid_baseline.parquet"
)

test_df = pd.read_parquet(
    "../data/modeling/test_baseline.parquet"
)

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(661318, 35)
(336669, 35)
(287447, 35)


# Create Features and Target

The preprocessing utility performs:

- Leakage removal
- Metadata removal
- Constant feature removal
- Date feature engineering

The output consists of:

- X = predictor variables
- y = stress_flag

In [4]:
X_train, y_train = prepare_xy(
    train_df
)

X_valid, y_valid = prepare_xy(
    valid_df
)

X_test, y_test = prepare_xy(
    test_df
)

print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)

(661318, 24)
(336669, 24)
(287447, 24)


# Build Preprocessing Pipeline

Machine learning models require numerical input.

The preprocessing pipeline performs:

Numerical Features:
- Median imputation
- Standard scaling

Categorical Features:
- Most frequent imputation
- One-hot encoding

The preprocessing pipeline is fitted only on the training dataset to prevent information leakage.

In [5]:
preprocessor = build_preprocessor()

In [6]:
preprocessor.fit(
    X_train
)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

# Transform Datasets

After fitting the preprocessing pipeline, all datasets are transformed into model-ready numerical representations.

These transformed matrices will be used for Logistic Regression training.

In [7]:
X_train_processed = (
    preprocessor.transform(
        X_train
    )
)

X_valid_processed = (
    preprocessor.transform(
        X_valid
    )
)

X_test_processed = (
    preprocessor.transform(
        X_test
    )
)

print(
    X_train_processed.shape
)

print(
    X_valid_processed.shape
)

print(
    X_test_processed.shape
)

(661318, 587)
(336669, 587)
(287447, 587)


In [8]:
X_train_processed

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 15871632 stored elements and shape (661318, 587)>

In [9]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)

len(feature_names)

587

In [10]:
pd.Series(
    feature_names
).value_counts().head()

numeric__credit_score              1
numeric__mortgage_insurance_pct    1
numeric__num_units                 1
numeric__cltv                      1
numeric__dti                       1
Name: count, dtype: int64

In [11]:
pd.Series(
    feature_names
).sample(20)

289                             categorical__msa_34980.0
239                             categorical__msa_30020.0
478                       categorical__property_state_GA
34                              categorical__msa_12220.0
514                       categorical__property_state_UT
526                        categorical__property_type_SF
317                             categorical__msa_37764.0
488                       categorical__property_state_MA
426                             categorical__msa_47220.0
265                             categorical__msa_32820.0
333                             categorical__msa_39150.0
556    categorical__servicer_name_AURORA FINANCIAL GR...
44                              categorical__msa_13140.0
505                       categorical__property_state_OK
374                             categorical__msa_42060.0
136                             categorical__msa_21340.0
563    categorical__servicer_name_LAKEVIEW LOAN SERVI...
110                            

In [12]:
feature_names = preprocessor.get_feature_names_out()

[
    f
    for f in feature_names
    if "super_conforming_flag" in f
][:20]

['categorical__super_conforming_flag_Y']

In [13]:
[
    f
    for f in feature_names
    if "program_indicator" in f
][:20]

['categorical__program_indicator_9',
 'categorical__program_indicator_F',
 'categorical__program_indicator_H']

In [14]:
[
    f
    for f in feature_names
    if "interest_only_indicator" in f
][:20]

[]

In [15]:
for col in [
    "super_conforming_flag",
    "program_indicator",
    "harp_indicator",
    "interest_only_indicator",
    "mortgage_insurance_cancellation_indicator"
]:
    print("\n", col)

    print(
        train_df[col]
        .value_counts(dropna=False)
        .head(20)
    )


 super_conforming_flag
super_conforming_flag
NaN    639733
Y       21585
Name: count, dtype: int64

 program_indicator
program_indicator
9    586551
H     71625
F      3142
Name: count, dtype: int64

 harp_indicator
harp_indicator
NaN    652655
Y        8663
Name: count, dtype: int64

 interest_only_indicator
interest_only_indicator
N    661318
Name: count, dtype: int64

 mortgage_insurance_cancellation_indicator
mortgage_insurance_cancellation_indicator
7    449033
N    183580
Y     28705
Name: count, dtype: int64


In [16]:
X_train, y_train = prepare_xy(train_df)

preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)

print(X_train_processed.shape)

(661318, 587)


In [17]:
len(
    preprocessor.get_feature_names_out()
)

587

In [18]:
[
    f
    for f in preprocessor.get_feature_names_out()
    if "vantagescore" in f.lower()
]

[]

In [19]:
[
    f
    for f in preprocessor.get_feature_names_out()
    if "servicer_name" in f
][:10]

['categorical__servicer_name_AMERIHOME MORTGAGE COMPANY, LLC',
 'categorical__servicer_name_AURORA FINANCIAL GROUP, INC.',
 'categorical__servicer_name_CALIBER HOME LOANS, INC.',
 'categorical__servicer_name_CITIZENS BANK, NA',
 'categorical__servicer_name_FIFTH THIRD BANK, NATIONAL ASSOCIATION',
 'categorical__servicer_name_FREEDOM MORTGAGE CORPORATION',
 'categorical__servicer_name_HOME POINT FINANCIAL CORPORATION',
 'categorical__servicer_name_JPMORGAN CHASE BANK, NATIONAL ASSOCIATION',
 'categorical__servicer_name_LAKEVIEW LOAN SERVICING, LLC',
 'categorical__servicer_name_MATRIX FINANCIAL SERVICES CORPORATION']

In [20]:
[
    f
    for f in preprocessor.get_feature_names_out()
    if "program_indicator" in f
]

['categorical__program_indicator_9',
 'categorical__program_indicator_F',
 'categorical__program_indicator_H']

# Baseline Logistic Regression

A Logistic Regression model is used as the initial benchmark for borrower stress prediction.

Reasons for selection:

- Interpretable
- Computationally efficient
- Widely used in credit risk modelling
- Provides a strong baseline before evaluating more complex machine learning models

Class imbalance is addressed using class weighting.

In [21]:
from sklearn.linear_model import LogisticRegression

In [26]:
log_reg = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

log_reg.fit(
    X_train_processed,
    y_train
)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [27]:
valid_pred = log_reg.predict(
    X_valid_processed
)

valid_prob = log_reg.predict_proba(
    X_valid_processed
)[:, 1]

In [28]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = pd.DataFrame({

    "Metric": [

        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Value": [

        accuracy_score(
            y_valid,
            valid_pred
        ),

        precision_score(
            y_valid,
            valid_pred
        ),

        recall_score(
            y_valid,
            valid_pred
        ),

        f1_score(
            y_valid,
            valid_pred
        ),

        roc_auc_score(
            y_valid,
            valid_prob
        )
    ]
})

results

,Metric,Value
0,Accuracy,0.665580
1,Precision,0.241182
2,Recall,0.668238
3,F1,0.354439
4,ROC-AUC,0.727588


In [29]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_valid,
    valid_pred
)

cm

array([[193172,  97244],
       [ 15345,  30908]])

In [30]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_valid,
        valid_pred
    )
)

              precision    recall  f1-score   support

           0       0.93      0.67      0.77    290416
           1       0.24      0.67      0.35     46253

    accuracy                           0.67    336669
   macro avg       0.58      0.67      0.56    336669
weighted avg       0.83      0.67      0.72    336669



In [31]:
feature_names = (
    preprocessor.get_feature_names_out()
)

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": log_reg.coef_[0]
})

coef_df = coef_df.sort_values(
    "coefficient",
    ascending=False
)

coef_df.head(20)

,feature,coefficient
479,categorical__property_state_GU,1.389173
31,categorical__msa_12054.0,1.147969
540,categorical__seller_name_HOME POINT FINANCIAL ...,1.063946
232,categorical__msa_29484.0,1.026688
28,categorical__msa_11694.0,0.936406
576,categorical__servicer_name_TRUIST BANK,0.921649
554,"categorical__seller_name_WELLS FARGO BANK, N.A.",0.879716
140,categorical__msa_21780.0,0.860439
575,categorical__servicer_name_SPECIALIZED LOAN SE...,0.851754
407,categorical__msa_45294.0,0.811437


In [32]:
coef_df[
    coef_df["feature"].str.contains(
        "credit_score"
    )
]

,feature,coefficient
0,numeric__credit_score,-0.56512


In [33]:
coef_df[
    coef_df["feature"].str.contains(
        "dti"
    )
]

,feature,coefficient
4,numeric__dti,0.206243


In [34]:
coef_df[
    coef_df["feature"].str.contains(
        "ltv"
    )
]

,feature,coefficient
3,numeric__cltv,0.30058
6,numeric__ltv,-0.24528


In [35]:
coef_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": log_reg.coef_[0]
})

coef_importance["abs_coef"] = (
    coef_importance["coefficient"]
    .abs()
)

coef_importance.sort_values(
    "abs_coef",
    ascending=False
).head(30)

,feature,coefficient,abs_coef
572,categorical__servicer_name_QUICKEN LOANS INC.,-1.940015,1.940015
479,categorical__property_state_GU,1.389173,1.389173
31,categorical__msa_12054.0,1.147969,1.147969
540,categorical__seller_name_HOME POINT FINANCIAL ...,1.063946,1.063946
232,categorical__msa_29484.0,1.026688,1.026688
28,categorical__msa_11694.0,0.936406,0.936406
576,categorical__servicer_name_TRUIST BANK,0.921649,0.921649
561,categorical__servicer_name_HOME POINT FINANCIA...,-0.880132,0.880132
554,"categorical__seller_name_WELLS FARGO BANK, N.A.",0.879716,0.879716
140,categorical__msa_21780.0,0.860439,0.860439
